# Modelagem Não Supervisionada

Sem labels pré-definidos, buscamos **estrutura latente** nos dados.

1. **PCA** — Análise de Componentes Principais nas causas de atraso (5 variáveis correlacionadas → eixos principais)
2. **K-Means** — Agrupamento de aeroportos com perfis de atraso semelhantes

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

from modules.data_loader import load_flights_clean

sns.set_style('whitegrid')

In [ ]:
df = load_flights_clean()
print(f"Dataset: {df.shape}")

---
## Parte 1 — PCA nas Causas de Atraso

As 5 colunas de causa de atraso (`AIR_SYSTEM_DELAY`, `SECURITY_DELAY`, `AIRLINE_DELAY`, `LATE_AIRCRAFT_DELAY`, `WEATHER_DELAY`) são correlacionadas e representam a decomposição do atraso total.

**Filtramos apenas voos atrasados** (`IS_DELAYED == 1`, ~1M registros) — nos demais, as 5 colunas são todas 0, o que tornaria o PCA trivial.

In [ ]:
delay_cause_cols = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
                    'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

delayed = df[df['IS_DELAYED'] == 1].copy()
print(f"Voos atrasados: {len(delayed):,}")

X_causes = delayed[delay_cause_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_causes)

print(f"\nEstatísticas das causas (voos atrasados):")
print(delayed[delay_cause_cols].describe().round(1))

In [ ]:
pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled)

# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(1, 6), pca.explained_variance_ratio_, color='steelblue', edgecolor='white')
axes[0].set_title('Variância Explicada por Componente')
axes[0].set_xlabel('Componente')
axes[0].set_ylabel('Variância Explicada (%)')
axes[0].set_xticks(range(1, 6))

cumulative = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, 6), cumulative, 'o-', color='coral')
axes[1].axhline(y=0.9, color='gray', linestyle='--', alpha=0.7, label='90%')
axes[1].set_title('Variância Explicada Acumulada')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Variância Acumulada')
axes[1].set_xticks(range(1, 6))
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nVariância explicada por componente:")
for i, (var, cum) in enumerate(zip(pca.explained_variance_ratio_, cumulative)):
    print(f"  PC{i+1}: {var:.4f} ({cum:.4f} acumulada)")

In [ ]:
# Heatmap dos loadings
loadings = pd.DataFrame(
    pca.components_.T,
    index=delay_cause_cols,
    columns=[f'PC{i+1}' for i in range(5)]
)

plt.figure(figsize=(8, 5))
sns.heatmap(loadings, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, center=0)
plt.title('Loadings do PCA — Causas de Atraso')
plt.tight_layout()
plt.show()

print("\nInterpretação dos componentes:")
for i in range(min(3, 5)):
    top = loadings[f'PC{i+1}'].abs().sort_values(ascending=False)
    print(f"\n  PC{i+1} ({pca.explained_variance_ratio_[i]:.1%} var):")
    for cause, val in top.head(3).items():
        print(f"    {cause}: {loadings.loc[cause, f'PC{i+1}']:.2f}")

In [ ]:
# Scatter PC1 vs PC2, colorido pela causa dominante
dominant_cause = delayed[delay_cause_cols].idxmax(axis=1)

# Sample para visualização (100k pontos)
sample_idx = np.random.default_rng(42).choice(len(X_pca), min(100_000, len(X_pca)), replace=False)

plt.figure(figsize=(10, 7))
for cause in delay_cause_cols:
    mask = dominant_cause.iloc[sample_idx] == cause
    if mask.sum() > 0:
        plt.scatter(X_pca[sample_idx[mask], 0], X_pca[sample_idx[mask], 1],
                    alpha=0.3, s=5, label=cause.replace('_DELAY', ''))

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('PCA — Causas de Atraso (colorido pela causa dominante)')
plt.legend(markerscale=5, fontsize=9)
plt.tight_layout()
plt.show()

---
## Parte 2 — K-Means em Perfis de Aeroporto

**Pergunta**: É possível agrupar aeroportos com perfis de atraso semelhantes?

Agregamos métricas por aeroporto de origem e aplicamos K-Means para identificar clusters de perfis operacionais.

In [ ]:
# Agregar por aeroporto
airport_profile = df.groupby('ORIGIN_AIRPORT').agg(
    avg_dep_delay=('DEPARTURE_DELAY', 'mean'),
    pct_delayed=('IS_DELAYED', 'mean'),
    avg_air_system=('AIR_SYSTEM_DELAY', 'mean'),
    avg_airline=('AIRLINE_DELAY', 'mean'),
    avg_late_aircraft=('LATE_AIRCRAFT_DELAY', 'mean'),
    avg_weather=('WEATHER_DELAY', 'mean'),
    avg_security=('SECURITY_DELAY', 'mean'),
    total_flights=('IS_DELAYED', 'count'),
    avg_distance=('DISTANCE', 'mean'),
).reset_index()

# Filtrar aeroportos com >= 1000 voos
airport_profile = airport_profile[airport_profile['total_flights'] >= 1000].copy()
print(f"Aeroportos com >= 1000 voos: {len(airport_profile)}")

# Features para clustering
cluster_features = ['avg_dep_delay', 'pct_delayed', 'avg_air_system', 'avg_airline',
                    'avg_late_aircraft', 'avg_weather', 'avg_distance']
X_airport = airport_profile[cluster_features].values

scaler_airport = StandardScaler()
X_airport_scaled = scaler_airport.fit_transform(X_airport)

In [ ]:
# Método do Cotovelo + Silhouette Score
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_airport_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_airport_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(K_range, inertias, 'o-', color='steelblue')
axes[0].set_title('Método do Cotovelo')
axes[0].set_xlabel('k (número de clusters)')
axes[0].set_ylabel('Inércia')

axes[1].plot(K_range, silhouettes, 'o-', color='coral')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('k (número de clusters)')
axes[1].set_ylabel('Score')

plt.tight_layout()
plt.show()

print("\nSilhouette scores:")
for k, s in zip(K_range, silhouettes):
    marker = " ←" if s == max(silhouettes) else ""
    print(f"  k={k}: {s:.4f}{marker}")

In [ ]:
# K-Means com k escolhido (baseado no cotovelo/silhouette — tipicamente k=3 ou k=4)
best_k = K_range[np.argmax(silhouettes)]
print(f"k escolhido (melhor silhouette): {best_k}")

km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
airport_profile['cluster'] = km_final.fit_predict(X_airport_scaled)

print(f"\nAeroportos por cluster:")
print(airport_profile['cluster'].value_counts().sort_index())

In [ ]:
# PCA 2D para visualização dos clusters
pca_airport = PCA(n_components=2)
X_airport_2d = pca_airport.fit_transform(X_airport_scaled)

# Mapear nomes dos aeroportos
code_to_name = df.drop_duplicates('ORIGIN_AIRPORT').set_index('ORIGIN_AIRPORT')['ORIGIN_AIRPORT_NAME']

plt.figure(figsize=(10, 7))
colors = ['steelblue', 'coral', 'teal', 'orange', 'purple']
for c in range(best_k):
    mask = airport_profile['cluster'] == c
    plt.scatter(X_airport_2d[mask, 0], X_airport_2d[mask, 1],
                c=colors[c % len(colors)], s=50, alpha=0.7, label=f'Cluster {c}')

plt.xlabel(f'PC1 ({pca_airport.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca_airport.explained_variance_ratio_[1]:.1%})')
plt.title('Clusters de Aeroportos (PCA 2D)')
plt.legend()
plt.tight_layout()
plt.show()

# Bar chart dos centroids
centroids = pd.DataFrame(
    scaler_airport.inverse_transform(km_final.cluster_centers_),
    columns=cluster_features
)
centroids.index = [f'Cluster {i}' for i in range(best_k)]

centroids.T.plot(kind='bar', figsize=(12, 5), edgecolor='white')
plt.title('Perfil Médio por Cluster (valores originais)')
plt.ylabel('Valor')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# Listar aeroportos por cluster
for c in range(best_k):
    cluster_airports = airport_profile[airport_profile['cluster'] == c].sort_values('total_flights', ascending=False)
    names = cluster_airports['ORIGIN_AIRPORT'].map(lambda x: f"{x} ({code_to_name.get(x, 'Desconhecido')})")
    avg_delay = cluster_airports['avg_dep_delay'].mean()
    pct_del = cluster_airports['pct_delayed'].mean() * 100
    
    print(f"\n{'='*60}")
    print(f"Cluster {c} — {len(cluster_airports)} aeroportos")
    print(f"  Atraso médio: {avg_delay:.1f} min | % atrasados: {pct_del:.1f}%")
    print(f"  Top 10: {', '.join(names.head(10).values)}")

## Conclusões

### PCA — Causas de Atraso
- As 5 causas de atraso podem ser resumidas em 2-3 componentes principais
- Os loadings revelam quais causas covariam — tipicamente AIRLINE_DELAY e LATE_AIRCRAFT_DELAY estão correlacionados (problemas operacionais), enquanto WEATHER_DELAY é mais independente
- A visualização PC1 vs PC2 mostra a separação entre tipos de atraso

### K-Means — Perfis de Aeroporto
- Os aeroportos se agrupam em perfis operacionais distintos (ex: hubs com alto volume e atraso moderado vs aeroportos menores)
- Os centroids revelam quais métricas diferenciam os clusters
- Aeroportos no mesmo cluster compartilham padrões similares de atraso

### Limitações
- PCA assume relações lineares — algumas interações podem ser perdidas
- K-Means assume clusters esféricos — DBSCAN seria alternativa para clusters irregulares
- Agregação por aeroporto perde variabilidade intra-aeroporto
- Dataset de 2015 apenas — perfis podem mudar com o tempo